In [2]:
import torchaudio.transforms as T
import torch, torchaudio, librosa, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import math, glob, json

def preprocess_raw_audio(audio_path, target_sr=22050, hop_length=1024):
    """
    Loads a raw audio file, downsamples/resamples to target_sr, converts to mono,
    extracts the MelSpectrogram, and converts it to normalized dB scale.
    """
    # 1. Load Audio
    waveform, sr = torchaudio.load(audio_path)

    # 2. Convert to Mono if Stereo
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)

    # 3. Resample if necessary
    if sr != target_sr:
        resampler = T.Resample(orig_freq=sr, new_freq=target_sr)
        waveform = resampler(waveform)

    # 4. Generate Mel Spectrogram
    # Match the n_fft, hop_length, and n_mels used in your dataset extraction pipeline
    mel_transform = T.MelSpectrogram(
        sample_rate=target_sr,
        n_fft=hop_length, # Common default, adjust if needed
        hop_length=hop_length,
        n_mels=100
    )
    mel_spec = mel_transform(waveform)

    # 5. Convert to Decibel Scale and Normalize
    spec_to_db = T.AmplitudeToDB(stype="power", top_db=80.0)
    spec_db = spec_to_db(mel_spec).squeeze(0) # Shape: [n_mels, n_frames]

    db_max = spec_db.max()
    db_min = spec_db.min()

    if db_max - db_min > 1e-5:
        spec_db_norm = (spec_db - db_min) / (db_max - db_min)
    else: spec_db_norm = torch.zeros_like(spec_db)

    return spec_db_norm

In [3]:
class CNN(nn.Module):
    def __init__(self, out_size = 64):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=out_size, kernel_size=3, padding=1)
        self.avg_pool = nn.AdaptiveAvgPool2d((1, None)) #preserve time domain, compress only freq domain
        self.bn1 = nn.BatchNorm2d(num_features=32)
        self.bn2 = nn.BatchNorm2d(num_features=out_size)

    def forward(self,x):
        x=x.unsqueeze(1)
        x = self.bn1(F.relu(self.conv1(x)))
        x = self.bn2(F.relu(self.conv2(x)))
        x = self.avg_pool(x)
        return x

class AttentionBlock(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        #Linear layers to project Q, K, V
        #Those are trainable parameters for a single head that we will tweak
        self.W_q = nn.Linear(input_dim, output_dim, bias=False)
        self.W_k = nn.Linear(input_dim, output_dim, bias=False)
        self.W_v = nn.Linear(input_dim, output_dim, bias=False)

    def forward(self, query, key, value):
        #project Q, K, V
        q_logits = self.W_q(query)
        k_logits = self.W_k(key)
        v_logits = self.W_v(value)

        attention, weights = self.scaled_dot_product_attention(q_logits, k_logits, v_logits)
        return attention, weights
        #We encapsulate the calculations explicitly which are done by each head. Q, K, V are projected independantly

    def scaled_dot_product_attention(self, q_logits, k_logits = None, v_logits = None):
        k_logits = k_logits if k_logits is not None else q_logits
        v_logits = v_logits if v_logits is not None else q_logits
        assert q_logits.size(-1) == k_logits.size(-1), "query and key must have the same embedding dimension"

        dim_k = q_logits.size(-1) #embed dimensions of key
        q_k = q_logits @ k_logits.transpose(-1, -2) / dim_k**0.5 # compute dot product to obtain similarity

        attn_weights = torch.softmax(q_k, dim=-1)

        #compute weighted sum of value vectors
        attention = attn_weights @ v_logits # attn = (bs, seq_len, embed_dim)
        return attention, attn_weights


class PositionalEmbedding(nn.Module):
    def __init__(self, embed_dim, max_len=5000):
        super().__init__()
        #create a matrix that represents positional encoding for each token
        pe = torch.zeros(max_len, embed_dim)
        nominator = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1) # (max_len, 1)
        div_term = torch.exp(torch.arange(0, embed_dim, 2).float() * (-math.log(10000.0) / embed_dim))

        pe[:, 0::2] = torch.sin(nominator * div_term)
        pe[:, 1::2] = torch.cos(nominator * div_term)

        pe=pe.unsqueeze(0)
        self.register_buffer('pe', pe, persistent=False)

    def forward(self,x):
        x = x + self.pe[:, :x.size(1), :]
        return x


class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads=1):
        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads" # required for consistensy so all heads have the same


        self.embed_per_head = embed_dim // num_heads
        self.heads = nn.ModuleList([AttentionBlock(input_dim=embed_dim, output_dim=self.embed_per_head) for _ in range(num_heads)])

        self.projection = nn.Linear(embed_dim, embed_dim, bias=False) #final projection of MHA



    def forward(self, query, key, value):
        attentions_list = []
        attention_weights_list = []

        for head in self.heads:
            attention, attention_weights = head(query, key, value) # for each head calculate its attention
            attentions_list.append(attention)
            attention_weights_list.append(attention_weights)

        #concatenate attention outputs and take avg of attn weights
        attentions, attention_weights = torch.cat(attentions_list, dim=2), torch.stack(attention_weights_list).mean(dim=0)
        return self.projection(attentions), attention_weights

class SoundSequenceAnalyzer(nn.Module):
    def __init__(self, time_window_size = 128, cnn_out = 64, freq_vec_len = 100, num_heads=1): #embed dim - vector length for a single frequency (n_bins)
        super().__init__()
        self.cnn_preprocessor = CNN(out_size = cnn_out)#extract local acoustic features

        self.input_projection = nn.Linear(in_features=cnn_out + 9, out_features=freq_vec_len, bias=False) #6 additional features for difficulty parameters

        self.positional_embedding = PositionalEmbedding(embed_dim=freq_vec_len, max_len=time_window_size)

        self.mha = MultiHeadAttention(embed_dim=freq_vec_len, num_heads=num_heads)

        self.ffn = nn.Sequential(
            nn.Linear(freq_vec_len, freq_vec_len * 4),
            nn.ReLU(),
            nn.Linear(freq_vec_len * 4, freq_vec_len),
        )

        self.norm1 = nn.LayerNorm(freq_vec_len)
        self.norm2 = nn.LayerNorm(freq_vec_len)



        #predict different aspects
        self.hit_classifier = nn.Linear(freq_vec_len, 1) #Is there a hit object at this frame
        #self.pos = nn.Linear(freq_vec_len, 2) #X, Y
        self.pos = CoordMDNHead(freq_vec_len)
        self.obj_type = nn.Linear(freq_vec_len, 3) #circle, slider, spinner
        self.obj_attributes = nn.Linear(freq_vec_len, 3)# Predicts end_x, end_y, attr_val
        self.curve_classifier = nn.Linear(freq_vec_len, 4) # 4 classes: L, B, P, C
        self.anchor_count = nn.Linear(freq_vec_len, 4) # Predicts 0, 1, 2, or 3 anchors
        self.anchor_coords = nn.Linear(freq_vec_len, 6) # Predicts 3 pairs of (x,y)

    def forward(self, spectrogram, difficulty): #(B, 1, n_mels, T)

        x = self.cnn_preprocessor(spectrogram) #(B, 64, 1, Time) 64 feature maps, freq compressed to 1
        x = x.squeeze(2) #(B, 64, Time)
        x = x.permute(0, 2, 1) #(B, Time, 64) put time first, so each "token" is a 64-dim vector


        x = self.input_projection(torch.cat((x, difficulty), dim=-1)) #(B, Time, embed_dim). Concatenate with difficulty to distingush between different difficulties of the map
        x = self.positional_embedding(x) #(B, Time, embed_dim)

        #Perform self attention. Each frame looks at every other frame.
        attended, weights = self.mha(x, x, x)

        x = self.norm1(x + attended) #residual connection + layer norm
        x = self.norm2(x + self.ffn(x)) #(B, Time, embed_dim)

        is_object = self.hit_classifier(x)
        #coords = torch.sigmoid(self.pos(x)) #commented out for testing so that coords don't stuck at 0.5 0.5 (likely cause is sigmoid function). Trying to use simple linear output
        coords = self.pos(x)
        obj_type = self.obj_type(x)
        obj_attr = self.obj_attributes(x)
        curve_class = self.curve_classifier(x)
        anchor_count = self.anchor_count(x)
        anchor_coords = self.anchor_coords(x)
        return is_object, coords, obj_type, obj_attr, curve_class, anchor_count, anchor_coords

class CoordMDNHead(nn.Module):
    def __init__(self, in_features, n_components=5):
        super().__init__()
        self.n_components = n_components
        self.fc = nn.Linear(in_features, n_components + n_components*2 + n_components*2)

    def forward(self, x):
        out = self.fc(x)
        pi_logits = out[..., :self.n_components]
        mu = torch.sigmoid(out[..., self.n_components:self.n_components*3])
        log_sigma = out[..., self.n_components*3:]

        mu = mu.view(*mu.shape[:-1], self.n_components, 2)
        log_sigma = log_sigma.view(*log_sigma.shape[:-1], self.n_components, 2)
        return pi_logits, mu, log_sigma



def mdn_sample(pi_logits, mu, log_sigma, temperature=1.0):
    """
    Stochastically samples coordinates from the MDN parameters to inject variety.
    temperature: >1.0 increases randomness, <1.0 makes it closer to standard flow.
    """
    # 1. Sample a mixture component index based on pi probabilities
    pi_dist = torch.distributions.Categorical(logits=pi_logits / temperature)
    component_idx = pi_dist.sample() # Shape: [N]

    # 2. Gather the corresponding mu and log_sigma parameters
    N, K, D = mu.shape  # Batch/Masked elements, Components, Dimensions (2)
    idx = component_idx.unsqueeze(-1).unsqueeze(-1).expand(N, 1, D)

    chosen_mu = mu.gather(-2, idx).squeeze(-2)
    chosen_log_sigma = log_sigma.gather(-2, idx).squeeze(-2)
    chosen_sigma = torch.exp(chosen_log_sigma)

    # 3. Sample from the selected normal distribution
    epsilon = torch.randn_like(chosen_mu)
    sampled_coords = chosen_mu + (epsilon * chosen_sigma * temperature)

    return sampled_coords

In [4]:
import json
import os

def json_to_osu(parsed_data, output_path, audio_path):
    """
    Converts a parsed dictionary/JSON back into a valid .osu file format.

    :param parsed_data: Dict containing metadata, general, difficulty, and hit_objects.
    :param output_path: Path where the output .osu file will be saved.
    """
    lines = []

    # 1. Write the mandatory file format header (defaulting to v14)
    lines.append("osu file format v14")
    lines.append("")  # Blank line after header

    # 2. Reconstruct Key-Value Sections
    # Order matters slightly for readability, though the game is flexible
    kv_sections = ["general", "metadata", "difficulty"]
    for section_key in kv_sections:
        if section_key in parsed_data and parsed_data[section_key]:
            # Capitalize section name to match standard .osu style (e.g., [General])
            lines.append(f"[{section_key.capitalize()}]")
            for key, value in parsed_data[section_key].items():
                lines.append(f"{key}: {value}")
            lines.append("")  # Blank line after section

    # Generate Timing Point via Audio
    y_audio, sr = librosa.load(audio_path)
    tempo, _ = librosa.beat.beat_track(y=y_audio, sr=sr)
    bpm = float(tempo[0]) if isinstance(tempo, np.ndarray) else float(tempo)
    beat_length = 60000.0 / bpm if bpm > 0 else 500.0

    lines.append("[TimingPoints]")
    # Format: time, beatLength, meter, sampleSet, sampleIndex, volume, uninherited, effects
    lines.append(f"0,{beat_length},4,2,1,60,1,0")
    lines.append("")

    # Reconstruct HitObjects
    if "hit_objects" in parsed_data and parsed_data["hit_objects"]:
        lines.append("[HitObjects]")
        for obj in parsed_data["hit_objects"]:
            x = max(0, min(512, int(obj.get("x", 256))))
            y = max(0, min(384, int(obj.get("y", 192))))
            time = int(obj.get("time", 0))
            obj_type = int(obj.get("type", 1)) # Extracted from the model's argmax(obj_type)

            # Base parameters
            parts = [str(x), str(y), str(time), str(obj_type), "0"]

            inv_curve_map = {0: 'L', 1: 'B', 2: 'P', 3: 'C'}

            if obj_type == 2: # Slider (Assuming argmax gave us 2 for bit flag 2)
                length = max(10.0, float(obj.get("length", 100.0)))
                curve_letter = inv_curve_map.get(int(obj.get("curve_class_idx", 0)), 'L')

                # Fetch structural coordinates array
                anchors = obj.get("anchor_coords", [])

                if len(anchors) > 0:
                    # Construct multi-node anchor string format (e.g., B|200:100|250:300)
                    anchor_strings = [f"{ax}:{ay}" for ax, ay in anchors]
                    curve_syntax = f"{curve_letter}|" + "|".join(anchor_strings)
                else:
                    # Traditional fallback endpoint if zero anchors were predicted
                    end_x = max(0, min(512, int(obj.get("end_x", x))))
                    end_y = max(0, min(384, int(obj.get("end_y", y))))
                    curve_syntax = f"{curve_letter}|{end_x}:{end_y}"

                parts.append(curve_syntax)
                parts.append("1") # Default standard repeat slider loop count
                parts.append(str(length))

            elif obj_type == 8: # Spinner
                end_time = time + max(100, int(obj.get("attr_val", 1000)))
                parts.append(str(end_time))

            lines.append(",".join(parts))
        lines.append("")

    with open(output_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print(f"Successfully generated map with {len(generated_hit_objects)} structural elements!")


In [5]:
def process_long_sequence(model, X, difficulty, window_size, device="cpu"):
    """
    Slices an audio track sequentially into window_size segments,
    evaluates them via the model, and joins the final predictions.
    """
    model.eval()
    model.to(device)

    # Ensure standard batch dimensions exist [B, Features, Time]
    if X.dim() == 2:
        X = X.unsqueeze(0)
    if difficulty.dim() == 2:
        difficulty = difficulty.unsqueeze(0)

    total_frames = X.shape[-1]
    num_chunks = total_frames // window_size

    # Storage arrays for collecting outputs across loops
    all_is_object = []
    all_pi_logits = []
    all_mu = []
    all_log_sigma = []
    all_obj_types = []
    all_obj_attrs = []
    all_curve_types = []
    all_anchor_count = []
    all_anchor_coords = []
    with torch.no_grad():
        for i in range(num_chunks):
            start = i * window_size
            end = start + window_size

            # Slice step
            X_chunk = X[..., start:end].to(device)
            diff_chunk = difficulty[:, start:end, :].to(device)

            # Evaluate forward pass
            outputs = model(X_chunk, diff_chunk)

            # Accommodates both 3-output and 4-output model versions cleanly

            is_obj_pred, mdn_out, obj_type_pred, obj_attr_pred, curve_type, anchor_count, anchor_coords = outputs
            all_obj_attrs.append(obj_attr_pred.cpu())


            pi_logits, mu, log_sigma = mdn_out

            # Keep on CPU to preserve VRAM during loop steps
            all_is_object.append(is_obj_pred.cpu())
            all_pi_logits.append(pi_logits.cpu())
            all_mu.append(mu.cpu())
            all_log_sigma.append(log_sigma.cpu())
            all_obj_types.append(obj_type_pred.cpu())
            all_curve_types.append(curve_type.cpu())
            all_anchor_coords.append(anchor_coords.cpu())
            all_anchor_count.append(anchor_count.cpu())

    # Concatenate chunk blocks back into standard continuous timelines (dim 1)
    full_is_object = torch.cat(all_is_object, dim=1)
    full_pi_logits = torch.cat(all_pi_logits, dim=1)
    full_mu = torch.cat(all_mu, dim=1)
    full_log_sigma = torch.cat(all_log_sigma, dim=1)
    full_obj_types = torch.cat(all_obj_types, dim=1)
    full_curve_types = torch.cat(all_curve_types, dim=1)
    full_anchor_count = torch.cat(all_anchor_count, dim=1)
    full_anchor_coords = torch.cat(all_anchor_coords, dim=1)
    mdn_recombined = (full_pi_logits, full_mu, full_log_sigma)

    if all_obj_attrs:
        full_obj_attrs = torch.cat(all_obj_attrs, dim=1)
        return (full_is_object, mdn_recombined, full_obj_types, full_obj_attrs, full_curve_types, full_anchor_count, full_anchor_coords)

    return full_is_object, mdn_recombined, full_obj_types


In [10]:
model = SoundSequenceAnalyzer(time_window_size=1024, freq_vec_len=256, num_heads=32)
pytorch_total_params = sum(p.numel() for p in model.parameters())
print(pytorch_total_params)

838254


In [18]:
import torch
import numpy as np
import json

# --- CONFIGURATION PARAMETERS ---
MODEL_WEIGHTS_PATH = "w_4k.pth"
INPUT_AUDIO_PATH = "audio.mp3"
OUTPUT_OSU_PATH = "generated_map.osu"

WINDOW_SIZE = 1024
HOP_LENGTH = 1024
SAMPLE_RATE = 22050
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


model = SoundSequenceAnalyzer(time_window_size=1024, freq_vec_len=256, num_heads=32)
state_dict = torch.load(MODEL_WEIGHTS_PATH, weights_only=True, map_location=torch.device('cpu'))
model.load_state_dict(state_dict)
model.eval()

# 2. Preprocess and Extract Features from Raw Input Audio
print(f"Preprocessing audio timeline: {INPUT_AUDIO_PATH}...")
X_full = preprocess_raw_audio(INPUT_AUDIO_PATH, target_sr=SAMPLE_RATE, hop_length=HOP_LENGTH)
n_frames = X_full.shape[-1]

# 3. Build a Mock Difficulty Sequence
# Your model expects difficulty conditioning vectors [B, Time, Diff_Features] along the timeline.
# Here we generate an array of 1s (or whatever feature baseline represents your target difficulty rating)

difficulty_values = {
    "HPDrainRate": 5.0,
    "CircleSize": 3.5,
    "OverallDifficulty": 8.5,
    "ApproachRate": 8.5,
    "SliderMultiplier": 1.7,
    "SliderTickRate": 1.0,
    "star_rating": 7.015145822185371,
    "aim_rating": 2.6742569831988363,
    "speed_rating": 2.151578056370707,
}

difficulty_full = (
    torch.tensor(list(difficulty_values.values()), dtype=torch.float32)
    .view(1, 1, -1)              # (1, 1, 9)
    .expand(1, n_frames, -1)     # (1, n_frames, 9)
)/10
# 4. Handle Padding to Match Windows Cleanly
remainder = n_frames % WINDOW_SIZE
if remainder != 0:
    pad_size = WINDOW_SIZE - remainder
    X_full = torch.nn.functional.pad(X_full, (0, pad_size))
    difficulty_full = torch.nn.functional.pad(difficulty_full, (0, 0, 0, pad_size))

# 5. Run the Timeline Processing via the Chunking Helper
print("Evaluating timeline predictions...")
outputs = process_long_sequence(
    model=model,
    X=X_full,
    difficulty=difficulty_full,
    window_size=WINDOW_SIZE,
    device=DEVICE
)

# 6. Unpack Outputs
is_object_pred = outputs[0]
pi_logits, mu, log_sigma = outputs[1]
obj_type_pred = outputs[2]
obj_attr_pred = outputs[3]
curve_p = outputs[4]
anchor_count_p = outputs[5]
anchor_coords_p = outputs[6]

# 7. Apply the Threshold and Reconstruct Mappings
mask = (is_object_pred > 0).squeeze(-1)

if mask.any():
    print(f"Stochastically sampling {mask.sum().item()} active elements...")
    coords_out = mdn_sample(pi_logits[mask], mu[mask], log_sigma[mask], temperature=0.85)

    pred_classes = torch.argmax(obj_type_pred[mask], dim=-1)
    masked_attributes = obj_attr_pred[mask]
    pred_curves = torch.argmax(curve_p[mask], dim=-1)
    pred_anchor_counts = torch.argmax(anchor_count_p[mask], dim=-1)
    masked_anchors = anchor_coords_p[mask]

    mask_idxs = torch.where(mask == True)
    ms = (mask_idxs[1].numpy() * (1000 * HOP_LENGTH) / SAMPLE_RATE).astype(int)

    generated_hit_objects = []
    for idx, time_ms in enumerate(ms):
        x = max(0, min(512, int(512 * coords_out[idx, 0].item())))
        y = max(0, min(384, int(384 * coords_out[idx, 1].item())))
        model_class = pred_classes[idx].item()

        obj = {"x": x, "y": y, "time": int(time_ms)}

        if model_class == 0:
            obj["type"] = 1
        elif model_class == 1:
            obj["type"] = 2
            obj["length"] = max(10.0, float(masked_attributes[idx, 2].item() * 1000.0))
            obj["curve_class_idx"] = int(pred_curves[idx].item())

            num_anchors = int(pred_anchor_counts[idx].item())
            obj["anchor_coords"] = []
            for j in range(num_anchors):
                ax = max(0, min(512, int(512 * masked_anchors[idx, j*2].item())))
                ay = max(0, min(384, int(384 * masked_anchors[idx, (j*2)+1].item())))
                obj["anchor_coords"].append((ax, ay))

            obj["end_x"] = max(0, min(512, int(512 * masked_attributes[idx, 0].item())))
            obj["end_y"] = max(0, min(384, int(384 * masked_attributes[idx, 1].item())))
        elif model_class == 2:
            obj["type"] = 8
            duration_ms = max(100, int(masked_attributes[idx, 2].item() * 1000.0))
            obj["endTime"] = int(time_ms + duration_ms)

        generated_hit_objects.append(obj)

    # 8. Define standard container metadata dictionary
    # (Since there's no original .json file for a brand new song, pass a standard metadata dict)
    data = {
        "general": {"AudioFilename": os.path.basename(INPUT_AUDIO_PATH), "Mode": 0},
        "metadata": {"Title": "AI Generated Map", "Artist": "Transformer Model", "Creator": "BeatmapAI"},
        "difficulty": {"HPDrainRate": 5, "CircleSize": 4, "OverallDifficulty": 6, "ApproachRate": 9, "SliderMultiplier": 1.4},
        "hit_objects": generated_hit_objects
    }

    # 9. Build into full .osu file format structure
    json_to_osu(data, OUTPUT_OSU_PATH, INPUT_AUDIO_PATH)
    print("Inference generation process fully completed!")
else:
    print("The model did not find appropriate trigger frames. Lower your selection threshold rules.")

Preprocessing audio timeline: audio.mp3...
Evaluating timeline predictions...
Stochastically sampling 708 active elements...
Successfully generated map with 708 structural elements!
Inference generation process fully completed!
